# PANDAS MASTERY — CHAPTERS 57–63

```
PANDAS MASTERY
Current Chapters: 57 (Sales Analytics Project), 58 (Fraud Data Prep),
                  59 (Employee Analytics), 60 (Professional Project Structure),
                  61 (Pandas Testing), 62 (Interview Mastery), 63 (Coding Interview)
Progress:         63 / 76 chapters
Completed:        Ch1-56
Remaining:        13 chapters
```

Seven chapters: two more full applied projects (sales analytics, fraud feature engineering with
real temporal-leakage-safe velocity features), an employee attrition analysis, professional
project structure, a real Pandas test suite, and interview preparation. Every number below was
executed and verified before this notebook was written.


In [ ]:
import pandas as pd
import numpy as np
print("pandas version:", pd.__version__)
print("numpy version :", np.__version__)


---
# CHAPTER 57 — Real Industry Project #3: Sales Analytics

## Business problem
Build a complete sales analytics pipeline: transactions -> daily/weekly/monthly aggregation ->
product/region analysis -> ranking, assembling `groupby`, `pivot_table`, `resample`, `rolling`,
and `rank` (Chapters 21, 28, 30, 32, 33) into one applied workflow.


In [ ]:
rng = np.random.default_rng(0)
dates = pd.date_range("2023-01-01", periods=180, freq="D")

transactions = pd.DataFrame({
    "date": rng.choice(dates, size=800),
    "region": rng.choice(["East", "West", "North", "South"], size=800),
    "product": rng.choice(["Widget", "Gadget", "Gizmo"], size=800),
    "amount": rng.uniform(20, 300, 800).round(2),
})
transactions = transactions.sort_values("date").reset_index(drop=True)
print(transactions.head())


### Expected Output (verified on pandas 3.0.2)
```
        date region product  amount
0 2023-01-01  South  Widget  204.39
1 2023-01-01  South   Gizmo  181.97
2 2023-01-01   West  Gadget   50.71
3 2023-01-01  North  Gadget   95.49
4 2023-01-01  South   Gizmo  203.56
```

## Daily, weekly, monthly aggregation via `resample()`


In [ ]:
daily = transactions.set_index("date").resample("D")["amount"].sum()
weekly = transactions.set_index("date").resample("W")["amount"].sum()
monthly = transactions.set_index("date").resample("ME")["amount"].sum()

print("Daily sales (first 5):\n", daily.head())
print("\nWeekly sales (first 5):\n", weekly.head())
print("\nMonthly sales:\n", monthly)


### Expected Output (verified on pandas 3.0.2)
```
Daily sales (first 5):
 date
2023-01-01     736.12
2023-01-02    1101.59
2023-01-03    1286.61
2023-01-04     680.23
2023-01-05     427.69
Freq: D, Name: amount, dtype: float64

Weekly sales (first 5):
 date
2023-01-01     736.12
2023-01-08    6632.30
2023-01-15    4880.43
2023-01-22    4251.11
2023-01-29    4548.33
Freq: W-SUN, Name: amount, dtype: float64

Monthly sales:
 date
2023-01-31    22284.96
2023-02-28    17330.40
2023-03-31    18142.04
2023-04-30    20568.50
2023-05-31    23962.06
2023-06-30    21147.96
Freq: ME, Name: amount, dtype: float64
```

This is a direct, layered aggregation hierarchy -- the same underlying transaction data
re-bucketed at three granularities with zero re-derivation, exactly Chapter 30's "resample is
groupby with a time-bucket key" lesson doing real analytical work: a business dashboard would
typically show all three views side by side (daily for operational monitoring, weekly for
short-term trend, monthly for board-level reporting) computed from this single line pattern.

## Product x Region breakdown via `pivot_table`-style `unstack()`


In [ ]:
product_region = transactions.groupby(["region", "product"])["amount"].sum().unstack()
print(product_region)


### Expected Output (verified on pandas 3.0.2)
```
product    Gadget     Gizmo    Widget
region
East      9332.94  10187.18   8346.99
North    11100.33   9831.98  11439.92
South    10704.53  11716.14  10526.14
West     11334.38  10973.98   7941.41
```

## Region ranking via `rank()`


In [ ]:
region_summary = transactions.groupby("region").agg(
    total=("amount", "sum"), count=("amount", "count"), avg=("amount", "mean")
)
region_summary["rank"] = region_summary["total"].rank(ascending=False, method="min")
print(region_summary.sort_values("rank"))


### Expected Output (verified on pandas 3.0.2)
```
            total  count         avg  rank
region
South   32946.81    200  164.734050   1.0
North   32372.23    204  158.687402   2.0
West    30249.77    211  143.363839   3.0
East    27867.11    185  150.633027   4.0
```

### Reading count vs total vs average together -- a real analytical nuance
South ranks #1 by total revenue (`32946.81`) with only `200` transactions, while West has
**more** transactions (`211`) but a **lower** total (`30249.77`) -- South's higher per-transaction
average (`164.73` vs West's `143.36`) is what drives its total-revenue lead despite fewer
transactions. Ranking purely by total (as done here) is a legitimate, common business choice,
but a report that shows only the rank column would hide this -- always surface `count` and `avg`
alongside `total` so the reader can see *why* a region ranks where it does, not just that it
does.

## 7-day rolling average -- smoothing daily noise into a trend signal


In [ ]:
rolling_7d = daily.rolling(7, min_periods=1).mean()
print(rolling_7d.head(10))


### Expected Output (verified on pandas 3.0.2)
```
date
2023-01-01     736.120000
2023-01-02     918.855000
2023-01-03    1041.440000
2023-01-04     951.137500
2023-01-05     846.448000
2023-01-06     765.540000
2023-01-07     785.914286
2023-01-08     947.471429
2023-01-09     938.238571
2023-01-10     760.818571
Freq: D, Name: amount, dtype: float64
```

`min_periods=1` (Chapter 32) lets the first 6 days show a **partial-window** average rather than
`NaN` -- appropriate here since we'd rather see an early, growing-sample trend line than a gap at
the very start of the reporting period; this is exactly the trade-off Chapter 32 flagged as a
judgment call, made explicitly here for a dashboard use case where an early signal (even if
based on fewer days) is more useful than nothing.

### Interview Insight
> "A region ranks #1 by total revenue. What additional context would you want before presenting
> this as 'our best-performing region'?"
Transaction count and average transaction value, at minimum -- a region can lead in total purely
from having more transactions (volume) or from higher-value transactions (average), and these
imply very different business strategies (acquire more customers vs. increase order value).
Verified above: South's #1 total-revenue rank comes from a higher per-transaction average with
*fewer* transactions than West, not from higher volume -- a materially different story than "our
biggest market."


---
# CHAPTER 58 — Real Industry Project #4: Fraud Data Preparation

## Business problem
Build fraud-detection features that are genuinely leakage-safe -- velocity (transactions in a
recent time window), time-since-previous-transaction, and a customer's-own-historical-average
comparison -- all computed using **only** information available strictly before each
transaction's own timestamp, directly applying Chapter 39/42's discipline to a domain where
temporal leakage is especially easy to introduce by accident.


In [ ]:
txns = pd.DataFrame({
    "customer_id": [1,1,1,1,2,2,3,3,3],
    "timestamp": pd.to_datetime([
        "2023-01-01 10:00","2023-01-01 10:05","2023-01-01 10:07","2023-01-02 09:00",
        "2023-01-01 12:00","2023-01-05 12:00",
        "2023-01-01 08:00","2023-01-01 08:02","2023-01-01 08:03",
    ]),
    "amount": [50, 60, 5000, 45, 100, 110, 20, 25, 3000],
})
txns = txns.sort_values(["customer_id", "timestamp"]).reset_index(drop=True)

# --- Velocity feature: count of this customer's transactions in the PAST 10 minutes, PAST ONLY ---
def count_recent_txns(group, window_minutes=10):
    times = group["timestamp"].values
    counts = np.zeros(len(group), dtype=int)
    for i in range(len(group)):
        window_start = times[i] - np.timedelta64(window_minutes, "m")
        counts[i] = ((times[:i] >= window_start) & (times[:i] < times[i])).sum()
    return pd.Series(counts, index=group.index)

txns["txns_last_10min"] = txns.groupby("customer_id", group_keys=False).apply(count_recent_txns, include_groups=False)
txns["time_since_prev_txn_sec"] = txns.groupby("customer_id")["timestamp"].diff().dt.total_seconds()
txns["hour_of_day"] = txns["timestamp"].dt.hour
txns["amount_vs_customer_avg_so_far"] = (
    txns["amount"] / txns.groupby("customer_id")["amount"].transform(lambda s: s.shift(1).expanding().mean())
)

print(txns)


### Expected Output (verified on pandas 3.0.2)
```
   customer_id           timestamp  amount  txns_last_10min  time_since_prev_txn_sec  hour_of_day  amount_vs_customer_avg_so_far
0            1 2023-01-01 10:00:00      50                0                      NaN           10                            NaN
1            1 2023-01-01 10:05:00      60                1                    300.0           10                       1.200000
2            1 2023-01-01 10:07:00    5000                2                    120.0           10                      90.909091
3            1 2023-01-02 09:00:00      45                0                  82380.0            9                       0.026419
4            2 2023-01-01 12:00:00     100                0                      NaN           12                            NaN
5            2 2023-01-05 12:00:00     110                0                 345600.0           12                       1.100000
6            3 2023-01-01 08:00:00      20                0                      NaN            8                            NaN
7            3 2023-01-01 08:02:00      25                1                    120.0            8                       1.250000
8            3 2023-01-01 08:03:00    3000                2                     60.0            8                     133.333333
```

### Reading the fraud signal these features actually surface -- customer 1's third transaction
Customer 1's transaction at `10:07` shows `txns_last_10min = 2` (two prior transactions within
the past 10 minutes -- a genuine velocity spike) **and** `amount_vs_customer_avg_so_far =
90.91` -- this single transaction's amount (`5000`) is roughly **91 times** this customer's own
average of their *prior* transactions (`(50+60)/2 = 55`). Both signals independently and
strongly suggest anomalous behavior, computed using **only** data that existed before `10:07` --
this is exactly the kind of feature combination a real fraud model would weight heavily, and
it's leakage-safe by construction since `count_recent_txns` explicitly slices `times[:i]`
(strictly before the current row) and `amount_vs_customer_avg_so_far` uses `shift(1)` before
`expanding()`, mirroring Chapter 39's proven pattern exactly.

### Why `include_groups=False` is required in the `.apply()` call
Recent pandas versions changed `groupby().apply()`'s default behavior around whether the
grouping column itself is included in what gets passed to the applied function -- passing
`include_groups=False` here suppresses a deprecation warning and matches the function's actual
intent (it operates on `timestamp`/`amount` within each group, not on `customer_id` itself,
which is already the grouping key). This is a small, current-Pandas-version detail worth being
aware of if adapting this exact pattern.

### Interview Insight
> "How would you build a 'transaction velocity' feature (transactions per customer in a recent
> time window) without leaking future information?"
For each transaction, count only that customer's **prior** transactions (strictly before the
current transaction's own timestamp) that fall within the target time window -- implemented
above by slicing `times[:i]` (everything before the current row's position within the sorted,
per-customer group) and filtering to the window relative to `times[i]`. The critical discipline:
never include the current transaction itself or any later one in that count, exactly mirroring
the `shift(1)` pattern from Chapter 39 applied to a windowed count instead of a cumulative sum.


---
# CHAPTER 59 — Real Industry Project #5: Employee Analytics

## Business problem
Analyze attrition rate, department-wise breakdowns, salary, and tenure -- a complete HR
analytics report built entirely from `groupby` and datetime arithmetic.


In [ ]:
employees = pd.DataFrame({
    "employee_id": range(1, 13),
    "department": ["Sales","Eng","Sales","Eng","Marketing","Eng","Sales","Support","Eng","Marketing","Support","Sales"],
    "salary": [65000,95000,72000,110000,58000,130000,68000,55000,105000,61000,52000,70000],
    "hire_date": pd.to_datetime(["2019-01-15","2018-03-20","2021-02-10","2017-06-01","2020-01-05",
                                    "2016-04-15","2022-01-10","2021-08-01","2019-11-15","2020-06-01",
                                    "2022-03-01","2021-05-15"]),
    "status": ["Active","Active","Terminated","Active","Terminated","Active","Active","Active","Active","Terminated","Active","Active"],
})

reference_date = pd.Timestamp("2023-07-01")
employees["tenure_years"] = ((reference_date - employees["hire_date"]).dt.days / 365.25).round(2)

attrition_rate = (employees["status"] == "Terminated").mean()
print("Overall attrition rate:", attrition_rate)

dept_attrition = employees.groupby("department")["status"].apply(lambda s: (s == "Terminated").mean())
print("\nDepartment-wise attrition rate:\n", dept_attrition)


### Expected Output (verified on pandas 3.0.2)
```
Overall attrition rate: 0.25

Department-wise attrition rate:
 department
Eng          0.00
Marketing    1.00
Sales        0.25
Support      0.00
Name: status, dtype: float64
```

### Reading the department breakdown -- the overall number hides an alarming outlier
The overall `25%` attrition rate sounds moderate -- but Marketing shows a **100%** attrition
rate (both Marketing employees in this dataset have left), while Engineering and Support show
**0%**. A report that only surfaces the blended `25%` figure would completely hide this -- the
overall number is being pulled down by three departments with zero-or-low attrition, masking a
department that has lost its entire staff. **This is precisely why Chapter 38's "always pair an
aggregate with a group-wise breakdown" lesson matters for HR analytics specifically** -- an
aggregate attrition rate alone can hide a genuinely urgent, department-specific retention crisis.

## Salary and tenure analysis


In [ ]:
dept_salary = employees.groupby("department")["salary"].agg(["mean", "median", "min", "max"])
print("Department salary analysis:\n", dept_salary)

tenure_summary = employees.groupby("department")["tenure_years"].mean().sort_values(ascending=False)
print("\nAvg tenure by department:\n", tenure_summary)


### Expected Output (verified on pandas 3.0.2)
```
Department salary analysis:
                 mean    median    min     max
department
Eng         110000.0  107500.0  95000  130000
Marketing    59500.0   59500.0  58000   61000
Sales        68750.0   69000.0  65000   72000
Support      53500.0   53500.0  52000   55000

Avg tenure by department:
 department
Eng          5.5475
Marketing    3.2850
Sales        2.6100
Support      1.6200
Name: tenure_years, dtype: float64
```

### Connecting the three tables into one HR narrative
Engineering has both the **highest average salary** (`110000`) and the **longest average
tenure** (`5.55 years`) *and* **zero attrition** -- a coherent, mutually-reinforcing pattern
(well-compensated, experienced staff who stay). Marketing has the **lowest tenure among
non-Support departments** (`3.29 years`) alongside its **100% attrition rate** -- while its
salary (`59500`) isn't the lowest in the dataset (Support is lower, at `53500`), the combination
of low tenure and total attrition suggests something specific to Marketing beyond just
compensation (e.g., management, workload, role clarity) is worth investigating directly, rather
than assuming a pay-driven explanation the salary data alone doesn't fully support.

### Interview Insight
> "Your company's overall attrition rate looks acceptable at 25%, but leadership is worried
> about retention. What would you check?"
A department-wise (or role-wise, tenure-band-wise) breakdown, not just the blended overall
rate -- an acceptable-looking aggregate can hide a department experiencing near-total attrition,
exactly demonstrated above where Marketing's 100% attrition is completely invisible in the
overall 25% figure. Cross-referencing attrition against salary and tenure by the same grouping
helps distinguish compensation-driven attrition from other causes (management, role fit,
market competition for that specific skill set).


---
# CHAPTER 60 — Professional Project Structure

## Concept
The Pandas-specific instance of the exact structure the NumPy course established for its own
projects -- `notebooks/` for exploration, `src/` for tested/reusable code, `tests/` mirroring
`src/`.

```
pandas-data-project/
+-- data/
|   +-- raw/               <- original, UNTOUCHED source files
|   +-- interim/            <- partially-cleaned intermediate outputs
|   +-- processed/           <- final, model/analysis-ready data
|
+-- notebooks/
|   +-- 01_data_inspection.ipynb
|   +-- 02_data_cleaning.ipynb
|   +-- 03_eda.ipynb
|   +-- 04_feature_engineering.ipynb
|
+-- src/
|   +-- ingestion.py          <- read_csv/read_sql wrappers, source-specific parsing
|   +-- validation.py          <- Chapter 50's validate_dataframe() and similar
|   +-- cleaning.py             <- Chapter 54's clean_customer_data()-style functions
|   +-- features.py              <- Chapter 39's leakage-labeled feature functions
|   +-- analysis.py               <- Chapter 55-59-style reusable analysis functions
|
+-- reports/                        <- generated summaries, figures
|
+-- tests/                           <- Chapter 61's test suite, one file per src/ module
|   +-- test_cleaning.py
|   +-- test_features.py
|
+-- requirements.txt
|
+-- README.md
```

## Why `data/raw/interim/processed/` matters specifically for Pandas projects
Never overwrite `raw/` -- it's your ground truth, and re-running the pipeline from scratch should
always be possible. `interim/` holds intermediate, partially-cleaned states (useful for
debugging "which step introduced this problem" without re-running the whole pipeline from raw
data). `processed/` holds the final, analysis/model-ready output -- typically saved as Parquet
(NumPy course connection: recall the NumPy course's own file-I/O chapter established Parquet's
schema-preservation advantage over CSV, which applies identically here).

## Notebook code vs production code, concretely
```python
# notebooks/02_data_cleaning.ipynb -- exploratory, cell-by-cell, throwaway once done
df = pd.read_csv("../data/raw/customers.csv")
df["age"] = pd.to_numeric(df["age"], errors="coerce")
# ... 20 more scattered cells testing different approaches ...

# src/cleaning.py -- promoted ONLY once the approach is settled, tested, documented
def clean_customer_ages(df: pd.DataFrame) -> pd.DataFrame:
    """Coerce age to numeric, clip impossible values to NaN. See tests/test_cleaning.py."""
    df = df.copy()
    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df["age"] = df["age"].where(df["age"].between(0, 120))
    return df
```

### Interview Insight
> "Why keep `data/raw/` completely untouched rather than cleaning files in place?"
Reproducibility -- if the pipeline needs to be re-run (a bug fix, a new team member, a
regulatory audit), it must be possible to start from the exact original data and reproduce the
exact same processed output deterministically. Overwriting raw files destroys that
ground-truth, making any downstream discrepancy impossible to fully diagnose or reproduce.


---
# CHAPTER 61 — Pandas Testing

## Concept
`pd.testing.assert_frame_equal`/`assert_series_equal` are Pandas' equivalent of the NumPy
course's `np.testing.assert_allclose` -- purpose-built comparison, not `==`. Plus schema
assertions and a genuine, passing test suite for one of this course's own functions.


In [ ]:
def clean_amounts(s):
    return pd.to_numeric(s, errors="coerce").fillna(0)


def test_clean_amounts_basic():
    result = clean_amounts(pd.Series(["10", "20", "bad", "30"]))
    expected = pd.Series([10.0, 20.0, 0.0, 30.0])
    pd.testing.assert_series_equal(result, expected)

def test_clean_amounts_empty():
    result = clean_amounts(pd.Series([], dtype=object))
    assert len(result) == 0

def test_clean_amounts_all_valid():
    result = clean_amounts(pd.Series(["1", "2", "3"]))
    assert result.sum() == 6.0


tests = [test_clean_amounts_basic, test_clean_amounts_empty, test_clean_amounts_all_valid]
for t in tests:
    try:
        t()
        print(f"PASS: {t.__name__}")
    except AssertionError as e:
        print(f"FAIL: {t.__name__} -- {e}")

# --- Schema assertions -- the standard, fast sanity checks before any pipeline step trusts data ---
df = pd.DataFrame({"id": [1, 2, 3], "amount": [10.0, 20.0, 30.0]})
assert df.shape[0] > 0
assert "id" in df.columns
assert df["id"].is_unique
print("\nSchema assertions passed")


### Expected Output (verified on pandas 3.0.2)
```
PASS: test_clean_amounts_basic
PASS: test_clean_amounts_empty
PASS: test_clean_amounts_all_valid

Schema assertions passed
```

### Why `pd.testing.assert_series_equal`, not `==` or `.equals()`
`pd.testing.assert_series_equal(result, expected)` checks values, dtype, index, and name all
together, and -- critically -- supports tolerance parameters (`rtol`/`atol`, exactly the NumPy
course's Chapter 27 floating-point lesson) for approximate numeric comparison, raising a
detailed, specific `AssertionError` describing exactly what differs if the check fails (which
row, which value, expected vs actual) rather than just a bare `True`/`False`. This detailed
failure output is precisely why it's the standard tool for Pandas test suites, not a manual
`result.equals(expected)` boolean check that gives no diagnostic detail when it fails.

### `test_clean_amounts_empty` -- the edge case that's easy to forget
An empty Series is a genuinely common real input (a filter that matches zero rows, an empty
API response) -- testing it explicitly catches bugs where a function assumes at least one row
exists (e.g., calling `.iloc[0]` unconditionally) and would crash or misbehave on legitimately
empty input, exactly the NumPy course's Chapter 64 "empty array" edge-case discipline, now
applied to Pandas functions.

### Interview Insight
> "Why use `pd.testing.assert_series_equal()` instead of `assert result.equals(expected)` in a
> test?"
`assert_series_equal` provides a detailed, specific failure message (exactly which values,
indices, or dtypes differ) when the assertion fails, plus built-in tolerance parameters for
approximate floating-point comparison -- `.equals()` returns only a bare boolean with no
diagnostic detail, forcing manual investigation to find what actually differs when a test fails.


---
# CHAPTER 62 — Pandas Interview Mastery

## Beginner
**What is Pandas?** A library for labeled, heterogeneous, tabular data -- `Series` (1D) and
`DataFrame` (2D) built around an `Index` used for alignment, layered on top of NumPy.

**`.loc` vs `.iloc`?** `.loc` is label-based and slice-inclusive of the endpoint; `.iloc` is
position-based and slice-exclusive, matching Python's standard convention.

**`head()` vs `info()`?** `head()` shows actual row values (the first N); `info()` shows
structural metadata (dtypes, non-null counts, memory) -- both are part of the standard first-look
workflow, answering different questions.

## Intermediate
**`merge()` vs `concat()`?** `merge()` matches rows by a key **value** across two tables
(relational join semantics); `concat()` stacks DataFrames by **label alignment** along an axis,
with no concept of matching by value.

**`agg()` vs `transform()`?** `agg()` reduces each group to one row; `transform()` returns a
result the same length as the input, broadcasting the group statistic back to every original
row -- the tool for group-relative feature engineering.

**What is categorical dtype, and when does it help?** A dtype storing small integer codes plus
a lookup table instead of repeated full strings -- a large, measurable memory/speed win
specifically for **low-cardinality-relative-to-row-count** columns; no benefit (and some
overhead) for high-cardinality/mostly-unique columns.

## Advanced
**How does Pandas alignment work?** Every operation between two Series/DataFrames matches
elements by **index label** first, not position -- mismatched labels produce `NaN` rather than
an error, and this label-matching is the foundation of `merge`, `groupby`, and `reindex`.

**How do you optimize a 20GB dataset?** Convert repetitive text columns to `category`; narrow
numeric dtypes where precision allows (`float64` -> `float32`); load only needed columns
(`usecols=`); process in `chunksize`-bounded pieces with cross-chunk accumulation via
`.add(fill_value=0)`; consider Parquet over CSV for both size and dtype-preserving I/O; beyond a
certain scale, consider Polars/DuckDB/Spark instead of Pandas entirely.

**Why is `apply()` often slower than expected?** It's fundamentally still calling a Python
function once per row/element under the hood -- the NumPy course's Chapter 1 lesson (vectorized
operations dispatch to compiled loops processing the whole array at once) applies identically
here; `apply()` doesn't get that benefit, and can even be slower than a plain Python loop
depending on implementation details, verified directly in this course's own Chapter 18.

## Expert
**"You receive 500 million transaction records. Pandas cannot comfortably load the entire
dataset. What would you do?"** Chunked ingestion with cross-chunk incremental aggregation
(Chapter 44); if the required computation genuinely can't be expressed as simple chunk-wise
accumulation, evaluate Polars (single-machine, faster, lower memory), DuckDB (SQL-native,
excellent for aggregation-heavy analytical queries directly against files), or Spark
(genuinely distributed, multi-machine scale) -- the decision hinges on whether the bottleneck is
single-machine memory (Polars/DuckDB solve this) or genuine computational scale beyond one
machine (Spark).

**"Your feature engineering pipeline works in a notebook but produces different results in
production."** Investigate, in order: (1) library version differences (`pd.__version__`
mismatch, exactly this course's own dtype/behavior findings across chapters); (2) whether the
notebook's cell execution order differs from the production script's linear order -- a notebook
re-run out of order can silently use stale intermediate state a linear script never would; (3)
whether any transformation implicitly depends on the full dataset being present (e.g., a
full-dataset statistic used for imputation) versus production processing data in smaller
batches or a different order.


---
# CHAPTER 63 — Pandas Coding Interview

Attempt each yourself before checking the solution below it.

## Problem 1 — Top-N per group
**Task:** Given a DataFrame of `(category, value)`, return the top-2 rows by `value` **within
each** category, without using `.apply()` (which would be the slow, natural-but-suboptimal
first instinct).


In [ ]:
df = pd.DataFrame({
    "category": ["A", "A", "A", "B", "B", "B"],
    "value": [10, 50, 30, 5, 25, 15],
})

# --- Attempt yourself before viewing the solution ---
def top_n_per_group(df, group_col, value_col, n=2):
    pass  # YOUR IMPLEMENTATION HERE


### Solution 1


In [ ]:
def top_n_per_group(df, group_col, value_col, n=2):
    return (
        df.sort_values(value_col, ascending=False)
          .groupby(group_col)
          .head(n)
          .sort_values([group_col, value_col], ascending=[True, False])
    )

result = top_n_per_group(df, "category", "value", n=2)
print(result)


### Expected Output (verified on pandas 3.0.2)
```
  category  value
1        A     50
2        A     30
4        B     25
5        B     15
```

**Evaluation notes:** Sorting once (`O(n log n)`), then `groupby().head(n)` (picks the first `n`
rows per group in the now-sorted order) is the standard, efficient idiom -- avoids the common
`.apply(lambda g: g.nlargest(n, value_col))` pattern, which works but is meaningfully slower for
large data since it re-sorts within each group separately rather than sorting once globally.

---

## Problem 2 — Detect and report near-duplicate rows
**Task:** Given a customer DataFrame, find groups of rows that share the same `email` after
lowercasing and stripping whitespace, but have different `customer_id`s -- likely duplicate
signups.


In [ ]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "email": ["Alice@Test.com", " alice@test.com", "bob@test.com", "carol@test.com"],
})

def find_near_duplicate_emails(df):
    pass  # YOUR IMPLEMENTATION HERE


### Solution 2


In [ ]:
def find_near_duplicate_emails(df):
    normalized = df["email"].str.strip().str.lower()
    dupe_mask = normalized.duplicated(keep=False)
    return df[dupe_mask].assign(normalized_email=normalized[dupe_mask])

result = find_near_duplicate_emails(customers)
print(result)


### Expected Output (verified on pandas 3.0.2)
```
   customer_id             email normalized_email
0            1    Alice@Test.com   alice@test.com
1            2   alice@test.com   alice@test.com
```

**Evaluation notes:** Normalizing (strip + lowercase) before checking duplication, rather than
checking duplication on the raw values, is exactly Chapter 16's string-cleaning lesson applied
as a genuine deduplication tool -- `keep=False` (Chapter 13) surfaces **all** rows involved in
each duplicate group, not just the second-and-later occurrences, which is what you want for a
report a human will review and decide how to merge/resolve.


---
## Chapters 57–63 Summary

- **Sales analytics project**: multi-granularity `resample()`, product x region pivoting, and a
  rank/count/average reading that revealed South's revenue lead came from higher per-transaction
  value, not higher volume -- verified with real numbers.
- **Fraud data prep**: leakage-safe velocity and historical-average features, verified to
  correctly flag a genuine anomalous transaction (customer 1's `5000` transaction: 2 recent
  transactions, ~91x their own prior average) using strictly past-only data.
- **Employee analytics**: department-wise attrition (verified: Marketing at 100% vs an overall
  25%) demonstrates why blended aggregates can hide urgent, group-specific problems.
- **Project structure**: `data/raw/interim/processed/`, notebook-vs-`src/` promotion -- the
  Pandas-specific instance of the NumPy course's own project architecture.
- **Testing**: `pd.testing.assert_series_equal` for detailed, diagnosable failures; empty-input
  edge cases as a standing test discipline; schema assertions as fast pre-pipeline sanity checks.
- **Interview mastery & coding interview**: the SQL/NumPy-style equivalence answers, plus two
  worked coding problems (`groupby().head(n)` for efficient top-N-per-group, and normalize-then-
  detect for near-duplicate discovery) -- both verified against real, checkable output.

---

## Mastery Exercises

**Exercise 1:** For the sales project's `region_summary`, explain why sorting by `rank` gives
the same row order as sorting by `total` descending directly -- when would that NOT be true?

**Exercise 2 (Prediction):** In the fraud features, predict what `amount_vs_customer_avg_so_far`
would show for a customer's very **first** transaction, and why.

**Coding Challenge:** Extend `top_n_per_group` (Chapter 63) to accept a `smallest=False`
parameter, returning the bottom-N per group when `True`, reusing the same sort-then-head
pattern rather than duplicating logic.

**Debugging Challenge:**
```python
employees["attrition_flag"] = employees["status"] == "Terminated"
dept_rate = employees.groupby("department")["attrition_flag"].sum()
```
A teammate expected `dept_rate` to show a *rate* (a proportion between 0 and 1) but got raw
counts instead. Explain the bug and the one-word fix.

**Interview Questions:**
1. Why does an overall attrition/error/failure rate sometimes hide a department- or
   segment-specific crisis?
2. What makes a velocity feature (transactions in a recent window) leakage-safe or leaky?
3. Why prefer `pd.testing.assert_series_equal` over `.equals()` in a test suite?
4. When would you recommend moving from Pandas to Polars, DuckDB, or Spark?

**Quiz:**
- Q1: True/False -- `groupby(...).head(n)` after a global sort is a valid, efficient way to get
  the top-n rows per group.
- Q2: Which chapter's discipline does normalizing text before duplicate-checking (Chapter 63,
  Problem 2) directly reuse?
- Q3: What does `data/raw/` specifically protect against, in the professional project
  structure?

---

 